#  Multi-Modal Deep Learning Framework for Road Damage Detection 
**GSSoC 2026 | DL-Simplified**

Models: EfficientNet-B0 | ResNet50 | YOLOv8n | Vision Transformer (ViT)

---

##  0. Install & Import Dependencies

In [ ]:
import subprocess, sys
packages = [
    'torch', 'torchvision', 'timm', 'ultralytics',
    'opencv-python', 'matplotlib', 'seaborn',
    'scikit-learn', 'pandas', 'numpy', 'tqdm',
    'grad-cam', 'Pillow'
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print(' All packages ready!')

In [ ]:
import os, json, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
import timm

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)

warnings.filterwarnings('ignore')

#  Reproducibility 
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'  Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU    : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

##  1. HuggingFace Token (silences download warnings)

> **Optional but recommended.** Get a free token at https://huggingface.co/settings/tokens  
> It removes the rate-limit warning and speeds up pretrained weight downloads.

In [ ]:
#  Paste your HuggingFace token below (read-only token is enough) 
# Leave as None to skip  timm will still download weights, just slower.
HF_TOKEN = "PLACE_YOUR_TOKEN_HERE"

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
    print(' HuggingFace token set  no more rate-limit warnings.')
else:
    # Suppress the noisy unauthenticated warning without a token
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'
    import logging
    logging.getLogger('huggingface_hub').setLevel(logging.ERROR)
    print('  No HF token  pretrained weights will still download (may be slower).')
    print('   To silence warnings permanently: set HF_TOKEN above.')

##  2. Git Remote Health-Check

Checks that your fork remote is configured correctly before you start training.

In [ ]:
import subprocess

#  Set your GitHub username here 
YOUR_GITHUB_USERNAME = 'Adhavan1801'   #  change this!
REPO_NAME            = 'DL-Simplified'         # upstream repo name
UPSTREAM_OWNER       = 'abhisheks008'           # original repo owner

def run_git(cmd, cwd=None):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    return result.stdout.strip(), result.stderr.strip()

# Detect repo root (walk up from notebook location)
nb_dir = Path(os.getcwd())
repo_root = nb_dir
for p in [nb_dir] + list(nb_dir.parents):
    if (p / '.git').exists():
        repo_root = p
        break

print(f' Repo root detected : {repo_root}')

#  Check current remotes 
remotes_out, _ = run_git('git remote -v', cwd=repo_root)
print('\n Current remotes:')
print(remotes_out or '  (none found)')

#  Auto-fix: ensure origin points to YOUR fork 
expected_origin = f'https://github.com/{YOUR_GITHUB_USERNAME}/{REPO_NAME}.git'
expected_upstream = f'https://github.com/{UPSTREAM_OWNER}/{REPO_NAME}.git'

if YOUR_GITHUB_USERNAME == 'YOUR_USERNAME_HERE':
    print('\n  Please set YOUR_GITHUB_USERNAME at the top of this cell!')
else:
    origin_out, _ = run_git('git remote get-url origin', cwd=repo_root)
    upstream_out, _ = run_git('git remote get-url upstream', cwd=repo_root)

    # Fix origin if wrong
    if origin_out != expected_origin:
        if origin_out:
            run_git(f'git remote set-url origin {expected_origin}', cwd=repo_root)
            print(f'\n origin updated  {expected_origin}')
        else:
            run_git(f'git remote add origin {expected_origin}', cwd=repo_root)
            print(f'\n origin added  {expected_origin}')
    else:
        print(f'\n origin is correct: {origin_out}')

    # Add upstream if missing
    if not upstream_out:
        run_git(f'git remote add upstream {expected_upstream}', cwd=repo_root)
        print(f' upstream added  {expected_upstream}')
    else:
        print(f' upstream is set : {upstream_out}')

    #  Show current branch 
    branch, _ = run_git('git branch --show-current', cwd=repo_root)
    print(f'\n Current branch : {branch}')

    #  Verify remotes after fix 
    remotes_fixed, _ = run_git('git remote -v', cwd=repo_root)
    print('\n Remotes after check:')
    print(remotes_fixed)

    print('\n Git remotes are properly configured!')
    print('   Push command to use after training:')
    print(f'   git push origin {branch}')

##  3. Configuration & Paths

In [ ]:
#  Update BASE_DIR to your local Dataset path 
BASE_DIR  = Path(r'G:\contribution\DL-Simplified\Multi-Modal Road Damage Detection\Dataset\RDD_SPLIT')
TRAIN_DIR = BASE_DIR / 'train'
VAL_DIR   = BASE_DIR / 'val'
TEST_DIR  = BASE_DIR / 'test'

MODEL_DIR = Path(r'G:\contribution\DL-Simplified\Multi-Modal Road Damage Detection\Model')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# YOLO label class names (RDD-2022 standard)
CLASS_NAMES = ['D00', 'D10', 'D20', 'D40']
# D00=Longitudinal Crack, D10=Transverse Crack, D20=Alligator Crack, D40=Pothole
NUM_CLASSES = len(CLASS_NAMES)

#  Training hyperparameters 
IMG_SIZE        = 224
BATCH_SIZE      = 16
EPOCHS          = 15
LR              = 1e-4

#  Early-stopping hyperparameters 
PATIENCE        = 5     # stop if val_loss doesn't improve for 5 epochs
MIN_DELTA       = 1e-4  # minimum improvement to count as 'better'

print('Paths set ')
for split, d in [('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)]:
    n = len(list((d/'images').glob('*.jpg'))) if (d/'images').exists() else 0
    print(f'  {split:6s}: {n:>5,} images')

##  4. Exploratory Data Analysis (EDA)

In [ ]:
def parse_labels(split_dir):
    """Parse YOLO .txt labels  DataFrame."""
    records = []
    label_dir = split_dir / 'labels'
    for f in label_dir.glob('*.txt'):
        country = f.stem.split('_')[0]
        lines = f.read_text().strip().split('\n')
        for line in lines:
            if not line: continue
            parts = line.split()
            cls = int(parts[0])
            records.append({'file': f.name, 'class_id': cls,
                            'class_name': CLASS_NAMES[cls] if cls < len(CLASS_NAMES) else f'cls{cls}',
                            'country': country})
    return pd.DataFrame(records)

train_df = parse_labels(TRAIN_DIR)
val_df   = parse_labels(VAL_DIR)
test_df  = parse_labels(TEST_DIR)

print('=== Label distribution ===')
print('TRAIN:\n', train_df['class_name'].value_counts().to_string())
print('\nVAL:\n',   val_df['class_name'].value_counts().to_string())
print('\nTEST:\n',  test_df['class_name'].value_counts().to_string())

In [ ]:
palette = {'D00': '#e74c3c', 'D10': '#3498db', 'D20': '#2ecc71', 'D40': '#f39c12'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (df, title) in zip(axes, [(train_df,'Train'), (val_df,'Val'), (test_df,'Test')]):
    counts = df['class_name'].value_counts().reindex(CLASS_NAMES, fill_value=0)
    bars = ax.bar(counts.index, counts.values,
                  color=[palette.get(c,'#95a5a6') for c in counts.index],
                  edgecolor='black', linewidth=0.7)
    ax.bar_label(bars, fmt='%d', fontsize=10, padding=3)
    ax.set_title(f'{title}  Class Distribution', fontweight='bold', fontsize=12)
    ax.set_xlabel('Damage Type'); ax.set_ylabel('Annotation Count')
    ax.set_ylim(0, counts.max() * 1.15)
    ax.grid(axis='y', alpha=0.3)

labels_legend = ['D00: Longitudinal Crack','D10: Transverse Crack','D20: Alligator Crack','D40: Pothole']
handles = [patches.Patch(color=c, label=l) for c, l in zip(palette.values(), labels_legend)]
fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=10, bbox_to_anchor=(0.5, -0.08))
plt.suptitle('RDD-2022 Dataset  Annotation Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
country_cls = train_df.groupby(['country','class_name']).size().unstack(fill_value=0)
country_cls.plot(kind='bar', ax=ax, color=list(palette.values()), edgecolor='black', linewidth=0.5)
ax.set_title('Country-wise Damage Distribution (Train)', fontweight='bold', fontsize=13)
ax.set_xlabel('Country'); ax.set_ylabel('Count')
ax.legend(title='Damage Type')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'eda_country_distribution.png', dpi=150)
plt.show()

In [ ]:
def show_annotated_samples(split_dir, n=6, title='Samples'):
    img_dir = split_dir / 'images'
    lbl_dir = split_dir / 'labels'
    imgs = list(img_dir.glob('*.jpg'))[:n]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    color_map = {0:'red', 1:'blue', 2:'green', 3:'orange'}
    for ax, img_path in zip(axes.flatten(), imgs):
        img = Image.open(img_path).convert('RGB')
        W, H = img.size
        ax.imshow(img)
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().split('\n'):
                if not line: continue
                c, cx, cy, bw, bh = map(float, line.split())
                c = int(c)
                x1 = (cx - bw/2) * W; y1 = (cy - bh/2) * H
                rect = patches.Rectangle((x1, y1), bw*W, bh*H,
                                          linewidth=2, edgecolor=color_map.get(c,'white'),
                                          facecolor='none')
                ax.add_patch(rect)
                ax.text(x1, y1-5, CLASS_NAMES[c] if c < len(CLASS_NAMES) else f'cls{c}',
                        color=color_map.get(c,'white'), fontsize=8, fontweight='bold')
        ax.set_title(img_path.name[:30], fontsize=7)
        ax.axis('off')
    plt.suptitle(title, fontweight='bold', fontsize=13)
    plt.tight_layout()
    plt.savefig(MODEL_DIR / f'eda_samples.png', dpi=120)
    plt.show()

show_annotated_samples(TRAIN_DIR, title='Train Samples with Annotations')

## 5. Dataset & DataLoaders

In [ ]:
class RDDDataset(Dataset):
    """
    Image-level classification dataset from YOLO labels.
    Label = class of the first annotation in the .txt file.
    Images with no annotations are skipped.
    """
    def __init__(self, split_dir, transform=None):
        self.img_dir  = split_dir / 'images'
        self.lbl_dir  = split_dir / 'labels'
        self.transform = transform
        self.samples   = []
        for img_path in sorted(self.img_dir.glob('*.jpg')):
            lbl_path = self.lbl_dir / (img_path.stem + '.txt')
            if not lbl_path.exists(): continue
            lines = [l for l in lbl_path.read_text().strip().split('\n') if l]
            if not lines: continue
            cls = int(lines[0].split()[0])
            if cls >= NUM_CLASSES: continue
            self.samples.append((img_path, cls))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

train_tfm = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    T.RandomRotation(15),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_tfm = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds = RDDDataset(TRAIN_DIR, train_tfm)
val_ds   = RDDDataset(VAL_DIR,   val_tfm)
test_ds  = RDDDataset(TEST_DIR,  val_tfm)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'Train : {len(train_ds):,} images')
print(f'Val   : {len(val_ds):,} images')
print(f'Test  : {len(test_ds):,} images')

##  6. Training Utilities (Early Stopping + Progress Bar)

In [ ]:
# 
# Early Stopping
# 
class EarlyStopping:
    """
    Stops training when val_loss hasn't improved for `patience` epochs.
    Also saves the best model weights automatically.
    """
    def __init__(self, patience=PATIENCE, min_delta=MIN_DELTA, save_path=None):
        self.patience    = patience
        self.min_delta   = min_delta
        self.save_path   = save_path
        self.best_loss   = float('inf')
        self.counter     = 0
        self.should_stop = False
        self.best_epoch  = 0

    def step(self, val_loss, model, epoch):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.counter    = 0
            self.best_epoch = epoch
            if self.save_path:
                torch.save(model.state_dict(), self.save_path)
            return ' saved'
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
            return f'no improvement ({self.counter}/{self.patience})'


# 
# Train one epoch  batch-level tqdm (inner bar)
# 
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
    epoch,
    epochs,
    global_bar
):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(imgs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)

        avg_loss = running_loss / total
        avg_acc  = correct / total

        # update single global progress bar
        global_bar.update(1)

        global_bar.set_postfix({
            "epoch": f"{epoch}/{epochs}",
            "loss": f"{avg_loss:.4f}",
            "acc":  f"{avg_acc:.4f}",
            "lr":   f"{optimizer.param_groups[0]['lr']:.2e}"
        })

    return avg_loss, avg_acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return running_loss / total, correct / total, all_preds, all_labels


# 
# Main train loop  epoch-level tqdm (outer bar) + early stopping
# 
def train_model(model, model_name, epochs=EPOCHS, lr=LR):

    model = model.to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs
    )

    save_path = MODEL_DIR / f"{model_name}_best.pt"

    stopper = EarlyStopping(
        patience=PATIENCE,
        min_delta=MIN_DELTA,
        save_path=save_path
    )

    history = {
        "train_loss": [],
        "train_acc":  [],
        "val_loss":   [],
        "val_acc":    []
    }

    print(f'\n{"="*65}')
    print(f'   Training: {model_name}')
    print(f'{"="*65}')

    t0 = time.time()

    # total iterations across ALL epochs
    total_iters = epochs * len(train_loader)

    # SINGLE tqdm BAR
    global_bar = tqdm(
        total=total_iters,
        desc=f"{model_name}",
        unit="it",
        ncols=110
    )

    for epoch in range(1, epochs + 1):

        # TRAIN
        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            DEVICE,
            epoch,
            epochs,
            global_bar
        )

        # VALIDATION
        val_loss, val_acc, _, _ = evaluate(
            model,
            val_loader,
            criterion,
            DEVICE
        )

        scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        # Early stopping
        status = stopper.step(val_loss, model, epoch)

        print(
            f"\nEpoch {epoch:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={val_acc:.4f} | "
            f"{status}"
        )

        if stopper.should_stop:
            print(f"\n Early stopping triggered at epoch {epoch}")
            break

    global_bar.close()

    total_time = time.time() - t0

    print(f"\n Training completed in {total_time/60:.2f} min")

    # load best weights
    model.load_state_dict(torch.load(save_path))

    return model, history


# 
# Plot training curves
# 
def plot_history(history, model_name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ep = range(1, len(history['train_loss']) + 1)
    ax1.plot(ep, history['train_loss'], label='Train', color='steelblue', marker='o', ms=4)
    ax1.plot(ep, history['val_loss'],   label='Val',   color='tomato',    marker='o', ms=4)
    ax1.set(title=f'{model_name}  Loss', xlabel='Epoch', ylabel='Loss')
    ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(ep, history['train_acc'], label='Train', color='steelblue', marker='o', ms=4)
    ax2.plot(ep, history['val_acc'],   label='Val',   color='tomato',    marker='o', ms=4)
    ax2.set(title=f'{model_name}  Accuracy', xlabel='Epoch', ylabel='Accuracy')
    ax2.legend(); ax2.grid(alpha=0.3)

    # Mark early-stop point if shorter than max epochs
    if len(ep) < EPOCHS:
        for ax in (ax1, ax2):
            ax.axvline(len(ep), color='grey', linestyle='--', alpha=0.6, label='Early stop')
            ax.legend()

    plt.suptitle(f'{model_name}  Training Curves', fontweight='bold')
    plt.tight_layout()
    plt.savefig(MODEL_DIR / f'{model_name}_training_curves.png', dpi=150)
    plt.show()


# 
# Full evaluation on test set
# 
def full_evaluation(model, model_name, loader=None):
    if loader is None:
        loader = test_loader
    criterion = nn.CrossEntropyLoss()
    t0 = time.time()
    loss, acc, preds, labels = evaluate(model, loader, criterion, DEVICE)
    latency = (time.time() - t0) / len(loader.dataset) * 1000  # ms/image

    p  = precision_score(labels, preds, average='weighted', zero_division=0)
    r  = recall_score(labels, preds, average='weighted', zero_division=0)
    f1 = f1_score(labels, preds, average='weighted', zero_division=0)

    print(f'\n{""*55}')
    print(f'    {model_name}  Test Results')
    print(f'{""*55}')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {p:.4f}')
    print(f'  Recall    : {r:.4f}')
    print(f'  F1-Score  : {f1:.4f}')
    print(f'  Latency   : {latency:.2f} ms / image')
    print()
    print(classification_report(labels, preds, target_names=CLASS_NAMES, zero_division=0))

    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set(title=f'{model_name}  Confusion Matrix', xlabel='Predicted', ylabel='True')
    plt.tight_layout()
    plt.savefig(MODEL_DIR / f'{model_name}_confusion_matrix.png', dpi=150)
    plt.show()

    return {'model': model_name, 'accuracy': acc, 'precision': p,
            'recall': r, 'f1': f1, 'latency_ms': latency}

print(' Training utilities with early-stopping + epoch-bar loaded.')

##  7. Model 1  EfficientNet-B0

In [ ]:
efficientnet = timm.create_model('efficientnet_b0', pretrained=True, num_classes=NUM_CLASSES)
print(f'EfficientNet-B0  |  Params: {sum(p.numel() for p in efficientnet.parameters())/1e6:.2f}M')

efficientnet, eff_history = train_model(efficientnet, 'EfficientNet_B0')
plot_history(eff_history, 'EfficientNet_B0')
eff_metrics = full_evaluation(efficientnet, 'EfficientNet_B0')

##  8. Model 2  ResNet50

In [ ]:
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
resnet.fc = nn.Linear(resnet.fc.in_features, NUM_CLASSES)
print(f'ResNet50  |  Params: {sum(p.numel() for p in resnet.parameters())/1e6:.2f}M')

resnet, res_history = train_model(resnet, 'ResNet50')
plot_history(res_history, 'ResNet50')
res_metrics = full_evaluation(resnet, 'ResNet50')

##  9. Model 3  YOLOv8n (Object Detection)

In [ ]:
from ultralytics import YOLO

data_yaml = f"""path: {BASE_DIR.as_posix()}
train: train/images
val: val/images
test: test/images

nc: {NUM_CLASSES}
names: {CLASS_NAMES}
"""
yaml_path = MODEL_DIR / 'rdd2022.yaml'
yaml_path.write_text(data_yaml)
print('data.yaml:')
print(data_yaml)

In [ ]:
yolo_model = YOLO('yolov8n.pt')

yolo_results = yolo_model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=640,
    batch=BATCH_SIZE,
    patience=PATIENCE,           # built-in early stopping for YOLO
    device=0 if DEVICE.type == 'cuda' else 'cpu',
    project=str(MODEL_DIR),
    name='YOLOv8n',
    exist_ok=True,
    verbose=True
)

yolo_val = yolo_model.val(data=str(yaml_path), split='test')

yolo_map50     = float(yolo_val.box.map50)
yolo_map5095   = float(yolo_val.box.map)
yolo_precision = float(yolo_val.box.mp)
yolo_recall    = float(yolo_val.box.mr)

print(f'\n{""*55}')
print('    YOLOv8n  Test Results')
print(f'{""*55}')
print(f'  mAP@0.5      : {yolo_map50:.4f}')
print(f'  mAP@0.5:0.95 : {yolo_map5095:.4f}')
print(f'  Precision    : {yolo_precision:.4f}')
print(f'  Recall       : {yolo_recall:.4f}')

yolo_f1 = 2*yolo_precision*yolo_recall / (yolo_precision + yolo_recall + 1e-8)

# Measure latency
sample_imgs = list((TEST_DIR/'images').glob('*.jpg'))[:50]
t0 = time.time()
for p in sample_imgs:
    yolo_model.predict(str(p), verbose=False)
yolo_latency = (time.time() - t0) / len(sample_imgs) * 1000
print(f'  Latency      : {yolo_latency:.2f} ms / image')

yolo_metrics = {
    'model': 'YOLOv8n',
    'accuracy':  yolo_map50,
    'precision': yolo_precision,
    'recall':    yolo_recall,
    'f1':        yolo_f1,
    'latency_ms': yolo_latency
}

##  10. Model 4  Vision Transformer (ViT-B/16)

In [ ]:
vit = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=NUM_CLASSES)
print(f'ViT-B/16  |  Params: {sum(p.numel() for p in vit.parameters())/1e6:.2f}M')

vit, vit_history = train_model(vit, 'ViT_B16', epochs=EPOCHS, lr=5e-5)
plot_history(vit_history, 'ViT_B16')
vit_metrics = full_evaluation(vit, 'ViT_B16')

##  11. Grad-CAM Visualisations

In [ ]:
#  Reload trained models from saved weights 
import timm
import torchvision.models as models
import torch.nn as nn

# -- EfficientNet-B0 --
efficientnet = timm.create_model('efficientnet_b0', pretrained=False, num_classes=NUM_CLASSES)
efficientnet.load_state_dict(torch.load(MODEL_DIR / 'EfficientNet_B0_best.pt', map_location=DEVICE))
efficientnet = efficientnet.to(DEVICE)
efficientnet.eval()
print(" EfficientNet-B0 loaded")

# -- ResNet50 --
resnet = models.resnet50(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, NUM_CLASSES)
resnet.load_state_dict(torch.load(MODEL_DIR / 'ResNet50_best.pt', map_location=DEVICE))
resnet = resnet.to(DEVICE)
resnet.eval()
print(" ResNet50 loaded")

# -- ViT-B/16 --
vit = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=NUM_CLASSES)
vit.load_state_dict(torch.load(MODEL_DIR / 'ViT_B16_best.pt', map_location=DEVICE))
vit = vit.to(DEVICE)
vit.eval()
print(" ViT-B/16 loaded")


In [ ]:
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import cv2

#  Output folder (all GradCAM images saved here) 
IMAGE_DIR = Path(r'G:\contribution\DL-Simplified\Multi-Modal Road Damage Detection\Images')
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
print(f' GradCAM images  {IMAGE_DIR}')

#  ImageNet stats (same as training) 
_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

_cam_preprocess = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(_MEAN, _STD),
])

CLASS_FULL = {
    0: 'D00  Longitudinal Crack',
    1: 'D10  Transverse Crack',
    2: 'D20  Alligator Crack',
    3: 'D40  Pothole',
}


def load_for_cam(img_path):
    """Return (rgb float32 HW3 in [0,1],  normalised tensor 13HW)."""
    pil    = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    rgb    = np.asarray(pil, dtype=np.float32) / 255.0
    tensor = _cam_preprocess(pil).unsqueeze(0).to(DEVICE)
    return rgb, tensor


def vit_reshape_transform(tensor, height=14, width=14):
    """
    timm ViT output: [B, 1 + num_patches, D]
    Drop [CLS] token, reshape to spatial [B, D, H, W] for GradCAM.
    """
    out = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
    return out.transpose(2, 3).transpose(1, 2)


def get_one_img_per_class(split_dir):
    """
    Scan split_dir labels and return one representative image per class.
    Returns dict  {class_id: Path}  sorted by class_id.
    """
    img_dir = split_dir / 'images'
    lbl_dir = split_dir / 'labels'
    selected = {}
    for p in sorted(img_dir.glob('*.jpg')):
        lp = lbl_dir / (p.stem + '.txt')
        if not lp.exists():
            continue
        lines = [l for l in lp.read_text().strip().split('\n') if l]
        if not lines:
            continue
        cls = int(lines[0].split()[0])
        if cls < NUM_CLASSES and cls not in selected:
            selected[cls] = p
        if len(selected) == NUM_CLASSES:
            break
    return dict(sorted(selected.items()))


# Pick one test image per damage class
CAM_IMAGES = get_one_img_per_class(TEST_DIR)
print('\n Representative test images selected:')
for cid, p in CAM_IMAGES.items():
    print(f'   Class {cid}  {CLASS_NAMES[cid]:4s}  ({CLASS_FULL[cid]})    {p.name}')
print('\n GradCAM setup complete.')


In [ ]:
def run_single_model_gradcam(model, model_name, target_layers,
                              images_dict, save_dir,
                              reshape_transform=None,
                              cam_cls=GradCAM):
    """
    Produce a 3-row figure for one model:
      Row 0 : Original image
      Row 1 : Activation heatmap  (jet colourmap)
      Row 2 : GradCAM overlay     (heatmap blended onto original)

    Saves  <save_dir>/gradcam_<model_name>.png
    Returns the saved Path.
    """
    n_cls   = len(images_dict)
    fig, axes = plt.subplots(3, n_cls, figsize=(5.2 * n_cls, 13))

    row_titles = ['Original Image', 'Activation Heatmap', 'GradCAM Overlay']
    col_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']   # D00/D10/D20/D40

    with cam_cls(model=model,
                 target_layers=target_layers,
                 reshape_transform=reshape_transform) as cam:

        for col, (cls_id, img_path) in enumerate(images_dict.items()):
            rgb, tensor = load_for_cam(img_path)
            grayscale   = cam(
                input_tensor=tensor,
                targets=[ClassifierOutputTarget(cls_id)]
            )[0]                                      # H  W float32

            overlay = show_cam_on_image(rgb, grayscale, use_rgb=True)

            heatmap_bgr = cv2.applyColorMap(
                np.uint8(255 * grayscale), cv2.COLORMAP_JET)
            heatmap_rgb = cv2.cvtColor(heatmap_bgr, cv2.COLOR_BGR2RGB)

            imgs = [rgb, heatmap_rgb, overlay]
            for row, img in enumerate(imgs):
                ax = axes[row, col]
                ax.imshow(img)
                ax.axis('off')
                if row == 0:          # column header
                    ax.set_title(
                        f'{CLASS_NAMES[cls_id]}\n{CLASS_FULL[cls_id]}',
                        fontsize=10, fontweight='bold',
                        color=col_colors[col], pad=6
                    )
                if col == 0:          # row label on left
                    ax.set_ylabel(
                        row_titles[row],
                        fontsize=11, fontweight='bold', labelpad=8
                    )

    cam_label = cam_cls.__name__
    fig.suptitle(
        f'{model_name}  {cam_label} Analysis\n'
        f'Highlighting which image regions drive each damage-class prediction',
        fontsize=13, fontweight='bold', y=1.01
    )
    plt.tight_layout()

    safe_name = model_name.lower().replace('-', '_').replace('/', '_').replace(' ', '_')
    save_path = save_dir / f'gradcam_{safe_name}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'     Saved    {save_path}')
    return save_path


print(' run_single_model_gradcam() defined.')


In [ ]:
print(' EfficientNet-B0  GradCAM ...')
run_single_model_gradcam(
    model             = efficientnet,
    model_name        = 'EfficientNet_B0',
    target_layers     = [efficientnet.conv_head],
    images_dict       = CAM_IMAGES,
    save_dir          = IMAGE_DIR,
    reshape_transform = None,
    cam_cls           = GradCAM,
)

In [ ]:
print(' ResNet50  GradCAM ...')
run_single_model_gradcam(
    model             = resnet,
    model_name        = 'ResNet50',
    target_layers     = [resnet.layer4[-1]],
    images_dict       = CAM_IMAGES,
    save_dir          = IMAGE_DIR,
    reshape_transform = None,
    cam_cls           = GradCAM,
)


In [ ]:
print(' ViT-B/16  EigenCAM ...')
run_single_model_gradcam(
    model             = vit,
    model_name        = 'ViT_B16',
    target_layers     = [vit.blocks[-1].norm1],
    images_dict       = CAM_IMAGES,
    save_dir          = IMAGE_DIR,
    reshape_transform = vit_reshape_transform,
    cam_cls           = EigenCAM,
)

In [ ]:
MODEL_CONFIGS = [
    # (display_name,   model_obj,    target_layers,               reshape_transform,     cam_class)
    ('EfficientNet-B0', efficientnet, [efficientnet.conv_head],   None,                  GradCAM),
    ('ResNet50',        resnet,       [resnet.layer4[-1]],         None,                  GradCAM),
    ('ViT-B/16',        vit,          [vit.blocks[-1].norm1],      vit_reshape_transform, EigenCAM),
]

n_models  = len(MODEL_CONFIGS)
n_classes = len(CAM_IMAGES)

fig, axes = plt.subplots(
    n_models, n_classes,
    figsize=(5.5 * n_classes, 5.0 * n_models),
    gridspec_kw={'hspace': 0.12, 'wspace': 0.04}
)

col_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

print(' Building cross-model GradCAM comparison grid ...')

for m_idx, (mname, model, tlayers, rtfm, cam_cls) in enumerate(MODEL_CONFIGS):
    with cam_cls(model=model,
                 target_layers=tlayers,
                 reshape_transform=rtfm) as cam:

        for c_idx, (cls_id, img_path) in enumerate(CAM_IMAGES.items()):
            rgb, tensor = load_for_cam(img_path)
            grayscale   = cam(
                input_tensor=tensor,
                targets=[ClassifierOutputTarget(cls_id)]
            )[0]
            overlay = show_cam_on_image(rgb, grayscale, use_rgb=True)

            ax = axes[m_idx, c_idx]
            ax.imshow(overlay)
            ax.axis('off')

            # Column header (top row only)
            if m_idx == 0:
                ax.set_title(
                    f'{CLASS_NAMES[cls_id]}\n{CLASS_FULL[cls_id]}',
                    fontsize=10, fontweight='bold',
                    color=col_colors[c_idx], pad=6
                )

            # Row label (left column only)
            if c_idx == 0:
                ax.set_ylabel(
                    mname, fontsize=11, fontweight='bold', labelpad=8
                )

    print(f'    {mname} done')

fig.suptitle(
    'Multi-Model GradCAM Comparison  Road Damage Detection\n'
    'Each cell: GradCAM/EigenCAM overlay showing class-discriminative regions',
    fontsize=13, fontweight='bold', y=1.01
)

cmp_path = IMAGE_DIR / 'gradcam_comparison_grid.png'
plt.savefig(cmp_path, dpi=150, bbox_inches='tight')
plt.show()
print(f' Saved    {cmp_path}')